# 02 · Cloud cover per chip (s2cloudless) + full-train S1 histogram

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/02_cloud_cover.ipynb)

Runs `scripts/compute_cloud_cover.py` on every chip of all four splits: `cloud_cover.csv` (chip_id, split, cloud_fraction, valid_s2_fraction), a bin table and an example figure. Then runs `scripts/visualize_sample.py --n 0` for the S1 water / not-water histogram over the full train split. Used later to evaluate models at real cloud levels; not used for training.

**CPU runtime** is enough (s2cloudless/LightGBM does not use the GPU). Expect roughly 30-45 min for ~446 chips on Colab's 2 vCPUs. Needs `00_download_data.ipynb` to have run first.

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the repo, or pull if it is already cloned.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/1ow1atency/floodfusion.git'
REPO_DIR = Path('/content/floodfusion')

if (REPO_DIR / '.git').exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
assert _exit_code == 0, 'git failed, see output above'
!git -C {REPO_DIR} log --oneline -1

Install the package with the `clouds` extra (s2cloudless).

In [ ]:
!pip install -q -e "{REPO_DIR}[clouds]"
assert _exit_code == 0, 'pip install failed, see output above'

Paths. No `--allow-missing` below: on the full data any missing file must be an error.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/floodfusion/data/sen1floods11'
OUT_DIR = Path('/content/drive/MyDrive/floodfusion/outputs')
CLOUD_CSV = OUT_DIR / 'cloud_cover.csv'
CLOUD_FIG = OUT_DIR / 'cloud_examples.png'
HIST_FIG = OUT_DIR / 'train_s1_hist.png'
OUT_DIR.mkdir(parents=True, exist_ok=True)

Cloud cover for all splits (prints the bin table at the end).

In [ ]:
!cd {REPO_DIR} && python scripts/compute_cloud_cover.py --root "{DATA_ROOT}" --out "{CLOUD_CSV}" --fig "{CLOUD_FIG}"
assert _exit_code == 0, 'compute_cloud_cover.py failed, see output above'

S1 histogram (water vs not water) over the full train split; counts are accumulated chip by chip in fixed bins.

In [ ]:
!cd {REPO_DIR} && python scripts/visualize_sample.py --root "{DATA_ROOT}" --split train --n 0 --stats configs/band_stats.json --out "{OUT_DIR}"
assert _exit_code == 0, 'visualize_sample.py failed, see output above'

Chips per split and cloud bin (from the CSV on Drive).

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))  # editable install needs a restart to be importable here
from floodfusion.clouds import format_bin_table, read_cloud_csv

print(format_bin_table(read_cloud_csv(CLOUD_CSV)))

Figures: 6 cloudiest + 3 clearest train chips with the cloud mask, and the full-train S1 histogram.

In [ ]:
from IPython.display import Image, display

for fig in (CLOUD_FIG, HIST_FIG):
    print(fig)
    display(Image(filename=str(fig)))